# LRURec：算法、评估与推荐

先运行 `python train.py --model lrurec`。本 Notebook 展示验证选出的模型，并与 ItemCF、LightFM 使用相同规则比较。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'src' / 'models' / 'lrurec.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user
from src.evaluate import write_comparison

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'lrurec_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model lrurec。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新训练 LRURec。')

## 1. 算法与当前参数

LRURec 用复数对角线性递推、残差前馈层与共享电影嵌入，编码高分历史并预测下一部电影。相邻同时间戳的预测标签不参与训练。

当前权重以长度 50 的窗口训练，在第 1–50 轮中按最低验证下一物品交叉熵选择第 50 轮。固定权重后，比较推理窗口 25、50、100，按验证 NDCG@10 选择 100。因此训练窗口为 50、推理窗口为 100；没有在长度 100 上重训。


In [2]:
print('Selection:', run['selection_metric'])
parameters = {**run['selected_parameters'], 'training_max_length': run['training_max_length']}
display(pd.DataFrame([parameters]))
search = pd.read_csv(TABLE_DIR / 'lrurec_context_search.csv')
display(search[['inference_length', 'Precision@10', 'Recall@10', 'NDCG@10', 'selected']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}', 'NDCG@10': '{:.4f}'}))


Selection: Training epoch by validation next-item cross entropy; inference length by validation NDCG@10


,n_items,hidden_size,num_blocks,max_length,dropout,epochs,training_max_length
0,3667,32,2,100,0.2,50,50


,inference_length,Precision@10,Recall@10,NDCG@10,selected
0,50.000000,10.14%,17.86%,0.1630,False
1,25.000000,10.04%,17.74%,0.1620,False
2,100.000000,10.16%,17.86%,0.1631,True


## 2. 三种算法比较

使用共同评估用户与完整未评分训练目录，先按用户计算再取平均。
Precision@10＝命中数 /10；Recall@10＝命中数 /高分目标数；NDCG@10 考虑命中位置。
未评分电影的偏好未知，不计为离线已知命中。结果代表当前参数与轮数搜索范围，不能说明算法的最佳表现。

In [3]:
write_comparison(TABLE_DIR)
comparison = pd.read_csv(TABLE_DIR / 'comparison_metrics.csv')
display(comparison[['algorithm', 'split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}'}))

,algorithm,split,users,Precision@10,Recall@10,NDCG@10
0,ItemCF,validation,5776,6.00%,7.91%,0.0847
1,ItemCF,test,5765,5.68%,7.18%,0.0779
2,LightFM,validation,5776,3.99%,4.04%,0.0497
3,LightFM,test,5765,3.92%,4.01%,0.0486
4,LRURec,validation,5776,10.16%,17.86%,0.1631
5,LRURec,test,5765,8.93%,15.83%,0.1440


## 3. 按用户比较推荐电影

修改 `USER_ID`，每种算法推荐 5 部历史未评分电影，展示标题与类型。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = []
for algorithm in ['itemcf', 'lightfm', 'lrurec']:
    result = recommend_for_user(USER_ID, stage='test', k=TOP_K, algorithm=algorithm)
    result.insert(0, 'algorithm', algorithm)
    recommendations.append(result[['algorithm', 'rank', 'title', 'genres']])
display(pd.concat(recommendations, ignore_index=True))

,algorithm,rank,title,genres
0,itemcf,1,Raiders of the Lost Ark (1981),Action|Adventure
1,itemcf,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,itemcf,3,"Shawshank Redemption, The (1994)",Drama
3,itemcf,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,itemcf,5,Pulp Fiction (1994),Crime|Drama
5,lightfm,1,Raiders of the Lost Ark (1981),Action|Adventure
6,lightfm,2,"Shawshank Redemption, The (1994)",Drama
7,lightfm,3,"Silence of the Lambs, The (1991)",Drama|Thriller
8,lightfm,4,"Godfather, The (1972)",Action|Crime|Drama
9,lightfm,5,"Fugitive, The (1993)",Action|Thriller


参考：[LRURec: Sequential Recommendation with Linear Recurrent Units (WSDM 2024)](https://arxiv.org/abs/2310.02367)；[官方代码](https://github.com/yueqirex/LRURec)。本项目独立实现核心结构，并适配统一时间划分与评估。